# COVID-19 Global Data Analysis
**Tools used:** Python · Pandas · NumPy  
**Dataset:** [Kaggle – COVID-19 Dataset](https://www.kaggle.com/datasets/imdevskp/corona-virus-report)  
**Data Coverage:** January 22, 2020 – July 27, 2020

---

### What this project covers
1. Loading the data
2. Cleaning the data (missing values, types, duplicates)
3. Exploratory analysis
4. Mortality and recovery rate analysis
5. Summary statistics

---
## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Libraries loaded successfully.')

---
## Step 2 — Load the Data

In [ ]:
# Load all four datasets
world    = pd.read_csv('worldometer_data.csv')
country  = pd.read_csv('country_wise_latest.csv')
day_wise = pd.read_csv('day_wise.csv')
full     = pd.read_csv('covid_19_clean_complete.csv')

print('Dataset shapes:')
print(f'worldometer_data        : {world.shape}')
print(f'country_wise_latest     : {country.shape}')
print(f'day_wise                : {day_wise.shape}')
print(f'covid_19_clean_complete : {full.shape}')

In [ ]:
# Preview the main dataset we will use most
world.head()

In [ ]:
# Check column names and data types
world.info()

---
## Step 3 — Data Cleaning

Before analyzing anything, we need to:
- Check for duplicate rows
- Find and fix missing values
- Fix data types
- Rename columns to make them easier to work with

### 3.1 — Check for Duplicates

In [ ]:
# Always work on a copy so the original data stays unchanged
world_clean = world.copy()

duplicate_count = world_clean.duplicated().sum()
print(f'Number of duplicate rows: {duplicate_count}')

### 3.2 — Find Missing Values

In [ ]:
# Count missing values in each column
missing_count = world_clean.isnull().sum()
missing_pct   = (missing_count / len(world_clean) * 100).round(1)
missing_report = pd.DataFrame({'Missing Values': missing_count, 'Percentage (%)': missing_pct})

# Show only columns that actually have missing values
missing_report[missing_report['Missing Values'] > 0].sort_values('Missing Values', ascending=False)

### 3.3 — Fix Missing Values

In [ ]:
# Columns like NewCases, NewDeaths → fill with 0
# Reason: if a country didn't report new cases that day, it means 0 new cases
fill_with_zero = ['NewCases', 'NewDeaths', 'NewRecovered', 'TotalDeaths', 'TotalRecovered', 'ActiveCases', 'Serious,Critical']

for col in fill_with_zero:
    if col in world_clean.columns:
        world_clean[col] = world_clean[col].fillna(0)

# Rate columns → fill with median
# Reason: median is safer than mean when data has extreme values
fill_with_median = ['Tot Cases/1M pop', 'Deaths/1M pop', 'Tests/1M pop', 'TotalTests']

for col in fill_with_median:
    if col in world_clean.columns:
        world_clean[col] = world_clean[col].fillna(world_clean[col].median())

# Text columns → fill with 'Unknown'
for col in ['Continent', 'WHO Region']:
    if col in world_clean.columns:
        world_clean[col] = world_clean[col].fillna('Unknown')

# Population → fill with median
if 'Population' in world_clean.columns:
    world_clean['Population'] = world_clean['Population'].fillna(world_clean['Population'].median())

# Confirm no more missing values
print(f'Total missing values after cleaning: {world_clean.isnull().sum().sum()}')

### 3.4 — Fix Data Types

In [ ]:
int_columns = ['TotalCases', 'NewCases', 'TotalDeaths', 'NewDeaths', 'TotalRecovered', 'NewRecovered', 'ActiveCases', 'TotalTests']

for col in int_columns:
    if col in world_clean.columns:
        world_clean[col] = world_clean[col].astype(int)

print('Data types after fixing:')
print(world_clean[int_columns].dtypes)

### 3.5 — Rename Columns (easier to type)

In [ ]:
world_clean = world_clean.rename(columns={
    'Country/Region'    : 'Country',
    'TotalCases'        : 'Total_Cases',
    'TotalDeaths'       : 'Total_Deaths',
    'TotalRecovered'    : 'Total_Recovered',
    'ActiveCases'       : 'Active_Cases',
    'TotalTests'        : 'Total_Tests',
    'Serious,Critical'  : 'Serious_Critical',
    'Tot Cases/1M pop'  : 'Cases_per_1M',
    'Deaths/1M pop'     : 'Deaths_per_1M',
    'Tests/1M pop'      : 'Tests_per_1M',
    'WHO Region'        : 'WHO_Region'
})

print('Columns after renaming:')
print(world_clean.columns.tolist())

### 3.6 — Add Calculated Columns

In [ ]:
# Case Fatality Rate = (Deaths / Total Cases) × 100
# This tells us: out of every 100 confirmed cases, how many died?
world_clean['CFR'] = np.where(
    world_clean['Total_Cases'] > 0,
    (world_clean['Total_Deaths'] / world_clean['Total_Cases'] * 100), 0
)

# Recovery Rate = (Recovered / Total Cases) × 100
world_clean['Recovery_Rate'] = np.where(
    world_clean['Total_Cases'] > 0,
    (world_clean['Total_Recovered'] / world_clean['Total_Cases'] * 100), 0
)

# Active Rate = (Active Cases / Total Cases) × 100
world_clean['Active_Rate'] = np.where(
    world_clean['Total_Cases'] > 0,
    (world_clean['Active_Cases'] / world_clean['Total_Cases'] * 100), 0
)

print('New columns added: CFR, Recovery_Rate, Active_Rate')
world_clean[['Country', 'Total_Cases', 'Total_Deaths', 'Total_Recovered', 'CFR', 'Recovery_Rate']].head(8)

---
## Step 4 — Exploratory Analysis

Now that the data is clean, let's explore it.

### 4.1 — Global Totals

In [ ]:
total_cases     = world_clean['Total_Cases'].sum()
total_deaths    = world_clean['Total_Deaths'].sum()
total_recovered = world_clean['Total_Recovered'].sum()
total_active    = world_clean['Active_Cases'].sum()
global_cfr      = round(total_deaths / total_cases * 100, 2)
global_rec_rate = round(total_recovered / total_cases * 100, 2)

print('=======================================')
print('     GLOBAL COVID-19 SUMMARY')
print('=======================================')
print(f'  Total Confirmed : {total_cases:>15,.0f}')
print(f'  Total Deaths    : {total_deaths:>15,.0f}')
print(f'  Total Recovered : {total_recovered:>15,.0f}')
print(f'  Active Cases    : {total_active:>15,.0f}')
print(f'  CFR             : {global_cfr:>14.2f} %')
print(f'  Recovery Rate   : {global_rec_rate:>14.2f} %')
print('=======================================')

### 4.2 — Top 10 Countries by Confirmed Cases

In [ ]:
top10_cases = (
    world_clean[['Country', 'Total_Cases', 'Total_Deaths', 'Total_Recovered', 'Active_Cases']]
    .sort_values('Total_Cases', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top10_cases.index += 1
top10_cases

### 4.3 — Top 10 Countries by Deaths

In [ ]:
top10_deaths = (
    world_clean[['Country', 'Total_Deaths', 'Total_Cases', 'CFR']]
    .sort_values('Total_Deaths', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top10_deaths.index += 1
top10_deaths

### 4.4 — Cases per Continent

In [ ]:
continent_summary = (
    world_clean[world_clean['Continent'] != 'Unknown']
    .groupby('Continent')
    .agg(
        Countries      = ('Country',         'count'),
        Total_Cases    = ('Total_Cases',      'sum'),
        Total_Deaths   = ('Total_Deaths',     'sum'),
        Total_Recovered= ('Total_Recovered',  'sum')
    )
    .sort_values('Total_Cases', ascending=False)
)

continent_summary['CFR (%)']          = (continent_summary['Total_Deaths']    / continent_summary['Total_Cases'] * 100)
continent_summary['Recovery Rate (%)']= (continent_summary['Total_Recovered'] / continent_summary['Total_Cases'] * 100)
continent_summary['Share of Cases (%)']= (continent_summary['Total_Cases']    / continent_summary['Total_Cases'].sum() * 100)

continent_summary

---
## Step 5 — Mortality & Recovery Rate Analysis

> **Case Fatality Rate (CFR)** = Deaths ÷ Confirmed Cases × 100  
> **Recovery Rate** = Recovered ÷ Confirmed Cases × 100  
>
> ⚠️ Note: CFR is not the true death rate. Countries that test more will detect mild cases and show a *lower* CFR — not because the virus is less deadly, but because more cases are counted.

### 5.1 — Filter to Meaningful Countries
Countries with very few cases can have extreme CFR values (e.g. 1 death out of 4 cases = 25% CFR). We filter to countries with at least 1,000 cases.

In [ ]:
significant = world_clean[world_clean['Total_Cases'] >= 1000].copy()

print(f'Countries with 1,000+ cases : {len(significant)} out of {len(world_clean)}')
print(f'These countries account for : {significant["Total_Cases"].sum() / world_clean["Total_Cases"].sum() * 100:.1f}% of global cases')

### 5.2 — Countries with Highest CFR

In [ ]:
highest_cfr = (
    significant[['Country', 'Total_Cases', 'Total_Deaths', 'CFR', 'Tests_per_1M']]
    .sort_values('CFR', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
highest_cfr.index += 1
highest_cfr

### 5.3 — Countries with Lowest CFR

In [ ]:
lowest_cfr = (
    significant[['Country', 'Total_Cases', 'Total_Deaths', 'CFR', 'Tests_per_1M']]
    .sort_values('CFR', ascending=True)
    .head(10)
    .reset_index(drop=True)
)
lowest_cfr.index += 1
lowest_cfr

### 5.4 — Countries with Highest Recovery Rate

In [ ]:
highest_recovery = (
    significant[['Country', 'Total_Cases', 'Total_Recovered', 'Recovery_Rate', 'CFR']]
    .sort_values('Recovery_Rate', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
highest_recovery.index += 1
highest_recovery

### 5.5 — Testing vs CFR Relationship

In [ ]:
# We split countries into 'high testing' and 'low testing' groups

median_testing = significant['Tests_per_1M'].median()

high_testing = significant[significant['Tests_per_1M'] >= median_testing]
low_testing  = significant[significant['Tests_per_1M'] <  median_testing]

print(f'Median tests per 1M population: {median_testing:,.0f}')
print()
print(f'HIGH testing group ({len(high_testing)} countries):')
print(f'  Average CFR          : {high_testing["CFR"].mean():.2f}%')
print(f'  Average Recovery Rate: {high_testing["Recovery_Rate"].mean():.2f}%')
print()
print(f'LOW testing group ({len(low_testing)} countries):')
print(f'  Average CFR          : {low_testing["CFR"].mean():.2f}%')
print(f'  Average Recovery Rate: {low_testing["Recovery_Rate"].mean():.2f}%')
print()
print('→ Countries testing more show a lower CFR.')
print('  This is because they detect mild/asymptomatic cases that low-testing countries miss.')

### 5.6 — Correlation Between Testing and CFR

In [ ]:
corr = significant[['Tests_per_1M', 'CFR', 'Recovery_Rate', 'Cases_per_1M', 'Deaths_per_1M']].corr()

print('Correlation Matrix:')
print(corr)

---
## Step 6 — Time-Series Analysis (day_wise.csv)

In [ ]:
# Clean the day_wise dataset
day_clean = day_wise.copy()
day_clean['Date'] = pd.to_datetime(day_clean['Date'])
day_clean = day_clean.sort_values('Date').reset_index(drop=True)

# Add CFR and Recovery Rate over time
day_clean['CFR']           = (day_clean['Deaths']    / day_clean['Confirmed'] * 100)
day_clean['Recovery_Rate'] = (day_clean['Recovered'] / day_clean['Confirmed'] * 100)

# Add 7-day rolling average for new cases (smooths out reporting gaps)
day_clean['New_Cases_7day_Avg'] = day_clean['New cases'].rolling(window=7).mean()

print(f'Date range: {day_clean["Date"].min().date()} to {day_clean["Date"].max().date()}')
day_clean[['Date','Confirmed','Deaths','Recovered','CFR','Recovery_Rate','New_Cases_7day_Avg']].tail(10)

In [ ]:
# How did CFR and Recovery Rate change from start to end?
first_week = day_clean.head(7)
last_week  = day_clean.tail(7)

print('First week of data (Jan 2020):')
print(f'  Avg CFR          : {first_week["CFR"].mean():.2f}%')
print(f'  Avg Recovery Rate: {first_week["Recovery_Rate"].mean():.2f}%')
print()
print('Last week of data (Jul 2020):')
print(f'  Avg CFR          : {last_week["CFR"].mean():.2f}%')
print(f'  Avg Recovery Rate: {last_week["Recovery_Rate"].mean():.2f}%')

In [ ]:
# Monthly breakdown of new cases
day_clean['Month'] = day_clean['Date'].dt.to_period('M')

monthly = (
    day_clean.groupby('Month')
    .agg(
        New_Cases     = ('New cases',     'sum'),
        New_Deaths    = ('New deaths',    'sum'),
        New_Recovered = ('New recovered', 'sum')
    )
)

monthly['Monthly_CFR'] = (monthly['New_Deaths'] / monthly['New_Cases'] * 100)
print('Monthly New Cases, Deaths, and Recoveries:')
monthly

---
## Step 7 — Summary Statistics

### 7.1 — Descriptive Statistics

In [ ]:
summary_cols = ['Total_Cases', 'Total_Deaths', 'Total_Recovered', 'CFR', 'Recovery_Rate', 'Cases_per_1M', 'Tests_per_1M']

world_clean[summary_cols].describe()

### 7.2 — CFR Percentile Breakdown

In [ ]:
# What CFR do most countries fall under?
percentiles = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
cfr_pct = significant['CFR'].quantile(percentiles)

print('CFR Percentile Breakdown (countries with 1,000+ cases):')
print('-' * 40)
for p, val in zip(percentiles, cfr_pct):
    print(f'{int(p*100):>3}th percentile : {val:.2f}%')

print()
print(f'Mean CFR   : {significant["CFR"].mean():.2f}%')
print(f'Median CFR : {significant["CFR"].median():.2f}%')
print(f'Std Dev    : {significant["CFR"].std():.2f}%')

### 7.3 — Recovery Rate Breakdown by Continent

In [ ]:
continent_rates = (
    world_clean[world_clean['Continent'] != 'Unknown']
    .groupby('Continent')
    .agg(
        Avg_CFR           = ('CFR',           'mean'),
        Avg_Recovery_Rate = ('Recovery_Rate',  'mean'),
        Min_CFR           = ('CFR',           'min'),
        Max_CFR           = ('CFR',           'max')
    )
    .round(2)
    .sort_values('Avg_CFR', ascending=False)
)

continent_rates

### 7.4 — Export Cleaned Data

In [ ]:
world_clean.to_csv('worldometer_clean.csv', index=False)
day_clean.to_csv('day_wise_clean.csv', index=False)

print('Exported:')
print('  worldometer_clean.csv')
print('  day_wise_clean.csv')

---
### Data Cleaning Summary
---

### Analysis Findings

1. **The Americas had the most cases.** North and South America together accounted for the largest share of global confirmed cases by mid-2020, led by the USA and Brazil.

2. **Countries that tested more showed lower CFR.** The high-testing group had a noticeably lower average CFR than the low-testing group. This is because more testing catches mild and asymptomatic cases — making the denominator larger, which lowers the rate.

3. **Recovery rates improved significantly over time.** In the first week (January 2020), recovery rates were very low. By July 2020 they had risen considerably, likely due to improved treatment, more experience managing the disease, and populations with milder cases being counted.

4. **July 2020 saw the highest monthly new case counts.** The monthly breakdown clearly shows case counts accelerating into mid-2020.

5. **Most countries' CFR falls between 1–5%.** The percentile table shows the median CFR is low, but outliers (countries with very high CFR) pull the average up.